# Splines Specified by Their Coefficients
This code plots a two-dimensional closed curve with Cartesian coordinates made of cubic splines parameterized by specific spline cefficients, with knots being indicated by small black dots. The vertical component is shown in green and the horizontal component is shown in blue; they are both periodic of identical period of length $6.$ A black-outlined yellow thumb corresponds to a curvilinear abscissa of the curve; the slider allows one to explore exactly one period of the closed curve.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.patches as patches
from matplotlib.path import Path
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
c1 = np.array([3, -15, 3, -3, 15, -3], dtype = float) # Horizontal coeffs
c2 = np.array([-11, 13, 7, 7, 13, -11], dtype = float) # Vertical coeffs
oversampling = 15 # Samples per unit length

# Cubic splines from spline coefficients
s1 = sk.PeriodicSpline1D.from_spline_coeff(c1, degree = 3)
s2 = sk.PeriodicSpline1D.from_spline_coeff(c2, degree = 3)
# Samples of the horizontal and vertical components
x1 = s1.get_samples(0, support_length = s1.period, oversampling = oversampling)
x2 = s2.get_samples(0, support_length = s2.period, oversampling = oversampling)
# Knots
knt1 = s1.get_knots()
knt2 = s2.get_knots()

# Plot
def update_plot (
    t = 2.5
):
    # Layout of the plot
    (fig, ax) = plt.subplots()
    fig.set_size_inches((6, 18))
    ax.set_aspect(1)
    ax.spines[:].set_color("none")
    plt.tick_params(
        bottom = False,
        labelbottom = False,
        left = False,
        labelleft = False
    )
    
    # Spline curve
    x = np.transpose([np.concatenate((x1, x1[0:1])), np.concatenate((x2, x2[0:1]))])
    ax.add_patch(patches.PathPatch(Path(x), edgecolor = "red", facecolor = "pink"))
    
    # Coordinates
    ax.add_patch(patches.Circle((0.0, 0.0), 0.25, color = "silver"))
    ax.add_patch(patches.PathPatch(
        Path([[0.0, 0.0], [s1.at(t), 0.0]]),
        fill = False,
        lw = 0.5,
        color = "darkblue"
    ))
    ax.add_patch(patches.PathPatch(
        Path([[s1.at(t), s2.at(t)], [s1.at(t), 0.0]]),
        fill = False,
        lw = 0.5,
        color = "green"
    ))
    ax.add_patch(patches.PathPatch(
        Path([[0.0, 0.0], [0.0, s2.at(t)]]),
        fill = False,
        lw = 0.5,
        color = "green"
    ))
    ax.add_patch(patches.PathPatch(
        Path([[s1.at(t), s2.at(t)], [0.0, s2.at(t)]]),
        fill = False,
        lw = 0.5,
        color = "darkblue"
    ))
    
    # Horizontal component
    h = np.linspace(start = -10.0, stop = 10.0, num = s1.period * oversampling)
    fh = s1.times(0.25).plus(13).get_samples(
        0,
        support_length = s1.period,
        oversampling = oversampling
    )
    ax.add_patch(patches.PathPatch(
        Path(np.transpose([h, fh])),
        fill = False,
        color = "darkblue"
    ))
    ax.add_patch(patches.PathPatch(
        Path([[-10, 13], [10, 13]]),
        fill = False,
        lw = 0.25,
        color = "black"
    ))
    
    # Vertical component
    v = np.linspace(start = -10.0, stop = 10.0, num = s2.period * oversampling)
    fv = s2.times(0.25).plus(17).get_samples(
        0,
        support_length = s2.period,
        oversampling = oversampling
    )
    ax.add_patch(patches.PathPatch(
        Path(np.transpose([v, fv])),
        fill = False,
        color = "green"
    ))
    ax.add_patch(patches.PathPatch(
        Path([[-10, 17], [10, 17]]),
        fill = False,
        lw = 0.25,
        color = "black"
    ))

    # Knots
    for knt in np.transpose([knt1, knt2]):
        ax.add_patch(patches.Circle(
            (s1.at(knt[0]), s2.at(knt[1])),
            0.125,
            color = "black"
        ))
        ax.add_patch(patches.Circle(
            ((20.0 / 6.0) * knt[0] - 10.0, s1.times(0.25).plus(13).at(knt[1])),
            0.125,
            color = "black"
        ))
        ax.add_patch(patches.Circle(
            ((20.0 / 6.0) * knt[0] - 10.0, s2.times(0.25).plus(17).at(knt[1])),
            0.125,
            color = "black"
        ))

    # Curvilinear thumb
    ax.add_patch(patches.Circle(
        (s1.at(t), s2.at(t)),
        0.25,
        facecolor = "yellow",
        edgecolor = "black"
    ))
    ax.add_patch(patches.Circle(
        ((20.0 / 6.0) * t - 10.0, s1.times(0.25).plus(13).at(t)),
        0.25,
        facecolor = "yellow",
        edgecolor = "black"
    ))
    ax.add_patch(patches.Circle(
        ((20.0 / 6.0) * t - 10.0, s2.times(0.25).plus(17).at(t)),
        0.25,
        facecolor = "yellow",
        edgecolor = "black"
    ))

    # Do plot the graph update
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 20)
    plt.show()

# Interactions
wide_floatslider_widget = widgets.FloatSlider(
    value = 2.5,
    min = 0.0,
    max = s1.period,
    step = 0.05,
    description = "t",
    layout = widgets.Layout(width = "5in")
)
widgets.interactive(
    update_plot,
    t = wide_floatslider_widget
)
